In [ ]:
%pip install optuna fastparquet --quiet

In [ ]:
import os
import sys
import glob
import shutil

SRC_ROOT = '/kaggle/input/mouse-dynamics-ic'
DST_ROOT = '/kaggle/working/mouse-dynamics'

os.makedirs(DST_ROOT, exist_ok=True)

shutil.copytree(
    f'{SRC_ROOT}/src',
    f'{DST_ROOT}/src',
    dirs_exist_ok=True
)

for src_db in glob.glob(f'{SRC_ROOT}/best-parameters/**/*.db', recursive=True):
    relative = os.path.relpath(src_db, f'{SRC_ROOT}/best-parameters')
    dst_db   = os.path.join(DST_ROOT, 'best-parameters', relative)
    os.makedirs(os.path.dirname(dst_db), exist_ok=True)
    if not os.path.exists(dst_db):
        shutil.copy2(src_db, dst_db)

print(f"src/     copiado : {os.path.exists(f'{DST_ROOT}/src')}")
print(f"db files copiados: {len(glob.glob(f'{DST_ROOT}/best-parameters/**/*.db', recursive=True))}")

In [ ]:
os.makedirs('/kaggle/working/datasets/raw', exist_ok=True)

for dataset in ['balabit', 'bogazici', 'minecraft']:
    src = f'/kaggle/input/datasets/mmoselli/mouse-dynamics-{dataset}/{dataset}'
    dst = f'/kaggle/working/datasets/raw/{dataset}'
    if not os.path.exists(dst):
        os.symlink(src, dst)
    print(f"{dataset:10s}: exists={os.path.exists(dst)}, symlink={os.path.islink(dst)}")

In [ ]:
os.chdir('/kaggle/working/mouse-dynamics')

if '/kaggle/working/mouse-dynamics' not in sys.path:
    sys.path.insert(0, '/kaggle/working/mouse-dynamics')

from src.classifiers    import EnumClassifiers
from src.dataset_loaders import EnumDatasets
from src.preprocessors  import EnumPreprocessors
from src.splitters      import EnumSplitters
from src.orchestrator   import Orchestrator

print("Imports OK")
print(f"  EnumClassifiers : {[e.value for e in EnumClassifiers]}")
print(f"  EnumDatasets    : {[e.value for e in EnumDatasets]}")
print(f"  EnumPreprocessors: {[e.value for e in EnumPreprocessors]}")
print(f"  EnumSplitters   : {[e.value for e in EnumSplitters]}")

In [ ]:
ALL_CLASSIFIERS  = [EnumClassifiers.RANDOM_FOREST, EnumClassifiers.MLP, EnumClassifiers.KNN]
ALL_WINDOW_SIZES = [10, 50, 100, 150, 200, 250]

total = len(ALL_CLASSIFIERS) * len(ALL_WINDOW_SIZES)
count = 0

for window_size in ALL_WINDOW_SIZES:
    count += len(ALL_CLASSIFIERS)
    print("=" * 80)
    print(f"[{count}/{total}] window_size={window_size}")
    print("=" * 80)

    orchestrator = Orchestrator(
        dataset=EnumDatasets.MINECRAFT,
        splitter=EnumSplitters.HALF,
        classifiers=ALL_CLASSIFIERS,
        preprocessor_window_size=window_size,
        preprocessor=EnumPreprocessors.KHAN,
        is_debug=False,
    )
    orchestrator.run()

    print(f"Concluído: window_size={window_size}\n")

print("=" * 80)
print("TODOS OS EXPERIMENTOS CONCLUÍDOS")
print("=" * 80)